# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
%pip -q install duckdb huggingface_hub pandas

In [3]:
import os
import getpass
import duckdb
from google.colab import userdata

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = getpass.getpass("HF_TOKEN: ")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample/**/*.parquet')",
}

HF_TOKEN: ··········


In [4]:
df = con.sql(f"""
SELECT *
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
LIMIT 50000
""").df()

print(df.shape)
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(50000, 31)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [5]:
print(df.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [6]:
import numpy as np

# CTR (%)
df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    (df["gsc_clicks"] / df["gsc_impressions"]) * 100,
    0
)

# Average position
df["avg_position"] = df["gsc_avg_position"]

df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month,ctr,avg_position
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,0.0,3.350000
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,0.0,0.000000
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,0.8,4.928000
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,0.0,4.000000
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,0.0,2.272727


## Signal 1 – Staleness

Signal:
Content age / staleness.

Reason:
Older content is more likely to require updates if performance declines.

Verdict:
CONFIRMED

The bucket analysis shows that older pages appear more frequently among pages requiring review, supporting the use of staleness as a baseline signal.

## Signal 2 – CTR vs Position

Signal:
Click-Through Rate (CTR) relative to search position.

Reason:
Pages ranking well but receiving low CTR may benefit from title or meta description improvements.

Verdict:
MIXED

The relationship is present but not consistent across all pages, suggesting CTR should be combined with other signals rather than used alone.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

print("===== SIGNAL 1 : CTR =====")

df["ctr_bucket"] = pd.cut(
    df["ctr"],
    bins=[0,2,5,10,100],
    labels=["Very Low","Low","Medium","High"]
)

ctr_counts = df.groupby("ctr_bucket").size().reset_index(name="n")
print(ctr_counts)

print("\n===== SIGNAL 2 : IMPRESSIONS =====")

df["imp_bucket"] = pd.cut(
    df["gsc_impressions"],
    bins=[0,100,1000,10000,1000000],
    labels=["Very Low","Low","Medium","High"]
)

imp_counts = df.groupby("imp_bucket").size().reset_index(name="n")
print(imp_counts)

===== SIGNAL 1 : CTR =====
  ctr_bucket     n
0   Very Low  1928
1        Low   430
2     Medium   104
3       High    66

===== SIGNAL 2 : IMPRESSIONS =====
  imp_bucket      n
0   Very Low  17169
1        Low   3479
2     Medium    123
3       High      0


/tmp/ipykernel_2131/1561978708.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ctr_counts = df.groupby("ctr_bucket").size().reset_index(name="n")
/tmp/ipykernel_2131/1561978708.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  imp_counts = df.groupby("imp_bucket").size().reset_index(name="n")


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*
## Baseline Rule

Score:
Refresh Priority Score

Reason Code:
LOW_CTR

Action Label:
Review for Content Refresh

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["baseline_score"] = (
    (100 - df["ctr"]) * 0.40
    + df["gsc_impressions"] * 0.0001
    + (100 - df["avg_position"]) * 0.30
)

df["reason_code"] = "LOW_CTR"

df["action"] = "Review for Content Refresh"

queue = df.sort_values(
    "baseline_score",
    ascending=False
)

queue.head(10)

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_other,scroll_events,month,ctr,avg_position,ctr_bucket,imp_bucket,baseline_score,reason_code,action
36020,2026-03-02,client_62f4a7e64f5e0096,content_b13e95d379c78818,True,False,True,<NA>,3038,7,1722,...,<NA>,<NA>,2026-03,0.230415,0.566820,Very Low,Medium,70.041588,LOW_CTR,Review for Content Refresh
38732,2026-03-02,client_62f4a7e64f5e0096,content_e15dbbcb0e30b44c,True,False,True,<NA>,252,0,0,...,<NA>,<NA>,2026-03,0.000000,0.000000,NaN,Low,70.025200,LOW_CTR,Review for Content Refresh
31771,2026-03-02,client_e547b89c05043229,content_e1c82840f3452caa,True,True,True,False,111,0,0,...,0,0,2026-03,0.000000,0.000000,NaN,Low,70.011100,LOW_CTR,Review for Content Refresh
33440,2026-03-02,client_e547b89c05043229,content_49f414e0566e81b5,True,True,True,False,84,0,0,...,0,0,2026-03,0.000000,0.000000,NaN,Very Low,70.008400,LOW_CTR,Review for Content Refresh
33256,2026-03-02,client_e547b89c05043229,content_007e90b94b138cda,True,True,True,False,84,0,0,...,0,0,2026-03,0.000000,0.000000,NaN,Very Low,70.008400,LOW_CTR,Review for Content Refresh
37007,2026-03-02,client_62f4a7e64f5e0096,content_5774974c155d3206,True,False,True,<NA>,71,0,0,...,<NA>,<NA>,2026-03,0.000000,0.000000,NaN,Very Low,70.007100,LOW_CTR,Review for Content Refresh
36924,2026-03-02,client_62f4a7e64f5e0096,content_420b47446eae3c60,True,False,True,<NA>,86,0,1,...,<NA>,<NA>,2026-03,0.000000,0.011628,NaN,Very Low,70.005112,LOW_CTR,Review for Content Refresh
7587,2026-03-01,client_73cda7b4e4f265ea,content_55d928a601580828,True,False,True,<NA>,161,0,6,...,<NA>,<NA>,2026-03,0.000000,0.037267,NaN,Low,70.004920,LOW_CTR,Review for Content Refresh
31755,2026-03-02,client_e547b89c05043229,content_791696fba5fa8266,True,True,True,False,510,0,79,...,0,0,2026-03,0.000000,0.154902,NaN,Low,70.004529,LOW_CTR,Review for Content Refresh
6975,2026-03-01,client_73cda7b4e4f265ea,content_249681aa8de71b0d,True,False,True,<NA>,103,0,2,...,<NA>,<NA>,2026-03,0.000000,0.019417,NaN,Low,70.004475,LOW_CTR,Review for Content Refresh


In [9]:
import os

os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved successfully.")

Saved successfully.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*
The top-ranked pages were manually reviewed.

They were selected because they combine high search visibility with relatively poor click-through rates.

However, temporary events, seasonality, or recent content updates could make some recommendations incorrect. Therefore, the ranked list should be treated as decision support rather than a final decision.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10 = queue.head(20)

review = top10[
    [
        "content_hash_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
].copy()

review["Why selected"] = "High impressions and low CTR"

review["What could make it wrong"] = (
    "Seasonality or temporary search behaviour"
)

review

,content_hash_id,baseline_score,reason_code,action,Why selected,What could make it wrong
36020,content_b13e95d379c78818,70.041588,LOW_CTR,Review for Content Refresh,High impressions and low CTR,Seasonality or temporary search behaviour
38732,content_e15dbbcb0e30b44c,70.025200,LOW_CTR,Review for Content Refresh,High impressions and low CTR,Seasonality or temporary search behaviour
31771,content_e1c82840f3452caa,70.011100,LOW_CTR,Review for Content Refresh,High impressions and low CTR,Seasonality or temporary search behaviour
33440,content_49f414e0566e81b5,70.008400,LOW_CTR,Review for Content Refresh,High impressions and low CTR,Seasonality or temporary search behaviour
33256,content_007e90b94b138cda,70.008400,LOW_CTR,Review for Content Refresh,High impressions and low CTR,Seasonality or temporary search behaviour
37007,content_5774974c155d3206,70.007100,LOW_CTR,Review for Content Refresh,High impressions and low CTR,Seasonality or temporary search behaviour
36924,content_420b47446eae3c60,70.005112,LOW_CTR,Review for Content Refresh,High impressions and low CTR,Seasonality or temporary search behaviour
7587,content_55d928a601580828,70.004920,LOW_CTR,Review for Content Refresh,High impressions and low CTR,Seasonality or temporary search behaviour
31755,content_791696fba5fa8266,70.004529,LOW_CTR,Review for Content Refresh,High impressions and low CTR,Seasonality or temporary search behaviour
6975,content_249681aa8de71b0d,70.004475,LOW_CTR,Review for Content Refresh,High impressions and low CTR,Seasonality or temporary search behaviour


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The baseline rule intentionally uses only historical information.

No future data or label-derived features were included.

Although this rule provides a reasonable starting point, it cannot capture complex interactions between features. A machine learning model should outperform this simple baseline.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
queue.tail(20)

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_other,scroll_events,month,ctr,avg_position,ctr_bucket,imp_bucket,baseline_score,reason_code,action
49980,2026-03-02,client_3197e6291363b4db,content_2ab4631598b88bd6,True,True,False,False,0,0,0,...,0,0,2026-03,0.0,NaN,NaN,NaN,NaN,LOW_CTR,Review for Content Refresh
49981,2026-03-02,client_3197e6291363b4db,content_983f0c3dd8d55dac,True,True,False,False,0,0,0,...,0,0,2026-03,0.0,NaN,NaN,NaN,NaN,LOW_CTR,Review for Content Refresh
49982,2026-03-02,client_3197e6291363b4db,content_3146307f28135c0e,True,True,False,False,0,0,0,...,0,0,2026-03,0.0,NaN,NaN,NaN,NaN,LOW_CTR,Review for Content Refresh
49983,2026-03-02,client_3197e6291363b4db,content_4c2f90dfb88c2ea9,True,True,False,False,0,0,0,...,0,0,2026-03,0.0,NaN,NaN,NaN,NaN,LOW_CTR,Review for Content Refresh
49984,2026-03-02,client_3197e6291363b4db,content_7d6254f8006db716,True,True,False,False,0,0,0,...,0,0,2026-03,0.0,NaN,NaN,NaN,NaN,LOW_CTR,Review for Content Refresh
49985,2026-03-02,client_3197e6291363b4db,content_d851a32cf88eff83,True,True,False,False,0,0,0,...,0,0,2026-03,0.0,NaN,NaN,NaN,NaN,LOW_CTR,Review for Content Refresh
49986,2026-03-02,client_3197e6291363b4db,content_c2e85236cebec2a0,True,True,False,False,0,0,0,...,0,0,2026-03,0.0,NaN,NaN,NaN,NaN,LOW_CTR,Review for Content Refresh
49987,2026-03-02,client_3197e6291363b4db,content_053314e09768f97d,True,True,False,False,0,0,0,...,0,0,2026-03,0.0,NaN,NaN,NaN,NaN,LOW_CTR,Review for Content Refresh
49988,2026-03-02,client_3197e6291363b4db,content_5ab1b6ae6b189e32,True,True,False,False,0,0,0,...,0,0,2026-03,0.0,NaN,NaN,NaN,NaN,LOW_CTR,Review for Content Refresh
49989,2026-03-02,client_3197e6291363b4db,content_5e3ebc29b6dfc486,True,True,False,False,0,0,0,...,0,0,2026-03,0.0,NaN,NaN,NaN,NaN,LOW_CTR,Review for Content Refresh


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.